# Intro to SQLAlchemy-Core (Basic)
This file demonstrates use of SQLAlchemy in a very similar way to use of psycopg2, it's a versatile tool!

First we handle our imports, os and dotenv aren't necessary for SQLAlchemy, but they're useful if you use a .env

In [ ]:
from os import getenv
from dotenv import load_dotenv
from sqlalchemy import create_engine, text, Row

Next we handle our globals.

SQLAlchemy, unlike psycopg2, is NOT a database driver. It does allow us to choose which SQL dialect we want and which driver we want to use with it. We will keep postgresql as our dialect and psycopg2, since we're used to them!

We also have to specify our username and password, which makes sense if we're creating an API to interact with our database. We only want authorized people to be able to read it and make changes. In a real project, you might receive the username and password and verify them.

We will also store the name of our database.

These all come together when we define **the engine.**

In [ ]:
SQL_DIALECT = "postgresql"
DIALECT_LIBRARY = "psycopg2" 

# TODO: Create a .env that contains your psql username and password
load_dotenv()
USER = getenv("USER")
PASSWORD = getenv("PASSWORD")

# TODO: Create a new database using psql named "test_sqlalchemy"
DATABASE = "test_sqlalchemy"

In [ ]:
engine = create_engine(f"{SQL_DIALECT}+{DIALECT_LIBRARY}://{USER}:{PASSWORD}@localhost/{DATABASE}")

With our engine set up, we can start executing SQL queries!

In [ ]:
def select_hello_word() -> str:
    """The simplest SQL query, returns 'hello world!'."""

    query = "SELECT 'hello world!';"

    with engine.connect() as conn:
        result = conn.execute(text(query))
        row = result.fetchone()

    return row[0]

print(select_hello_word())

Before we start playing with our own database, we'll need a function that sets up the tables. Fortunately with SQLAlchemy it's very easy to read and execute .sql files, so we will simply run our schema.sql!

In [ ]:
def reset_database() -> None:
    """Runs the schema.sql file to reset the database"""
    with open("schema.sql", "r", encoding="UTF-8") as schema:
        stmt = schema.read()
    with engine.connect() as conn:
        conn.execute(text(stmt))
        conn.commit()

reset_database()

With all the set up done, we'll make functions that use postgresql code to insert into, and select from, our database!

In [ ]:
def insert_lecturer(name: str) -> Row:
    """
    A query that inserts a lecturer into lecturer table given a name.
    Returns inserted row.
    """

    # Note slightly different syntax for parameterization
    # Parameters are preceded by colons
    query = """
            INSERT INTO lecturer (lecturer_name)
            VALUES (:lecturer_name)
            RETURNING *;
            """

    with engine.connect() as conn:
        # Similar to psycopg2 we pass parameters with execution
        result = conn.execute(text(query),{"lecturer_name": name})
        row = result.fetchone()
        conn.commit()

    return row


def select_lecturers() -> list[Row]:
    """Simple query to output all lecturers"""

    query = "SELECT * FROM lecturer;"

    with engine.connect() as conn:
        result = conn.execute(text(query))
        rows = result.fetchall()

    return rows

Now we can use these functions to populate our lecturers table and read from it!

In [ ]:
sigma_bot = insert_lecturer("SigmaBot")

# Each row is similar to a namedtuple,
# we can access their attributes directly
print(f"\nYou've just inserted {sigma_bot.lecturer_name}!")

select_lecturers()

In [ ]:
insert_lecturer("Siggy")
insert_lecturer("Ciggy??")

for lecturer in select_lecturers():
    print(lecturer.lecturer_name)